# Deep learning on text

## Load modules from repo

In [12]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [13]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [14]:
import src
from src.preprocessing.core import load_reproducible_split
# from src.preprocessing.pipelines.deep_learning_on_text import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_text.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

In [15]:
import importlib
importlib.reload(src.preprocessing.core)
# importlib.reload(src.preprocessing.pipelines.deep_learning_on_text)
importlib.reload(src.models.on_text.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_text_and_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_text_and_images/deep_learning.py'>

## Load tensorflow

In [16]:
import tensorflow as tf

In [17]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [18]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [19]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')

In [20]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [21]:
version=1
artifacts_folder=Path(f'artifacts/on_text/deep_learning/v{version}')
log_file_path=artifacts_folder / 'experiments.parquet'

fit_preprocessors=True
# fit_preprocessors=False

small_train_sample = False
# small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = False  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

augment=True  # Augment data for training

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False

## Preprocessing (TODO)

In [22]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [23]:
if rebalance_with_weights:
    print('using class weights')
else:
    print('not using class weights')

not using class weights


In [24]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

In [25]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

In [26]:
y_train.value_counts().describe()

count      27.000000
mean     2516.000000
std      1682.489662
min       611.000000
25%      1241.000000
50%      2137.000000
75%      3813.500000
max      8167.000000
Name: count, dtype: float64

In [27]:
print(X_train.shape)

(67932, 31)


In [ ]:
X_train.iloc[:,:2]  #TODO: preprocess designation, description

,designation,description
1887,Porte Bébé Violet Et Rouge Trois-En-Un Mère Mu...,Porte bébé Violet et rouge Trois-en-un mère mu...
70389,Jesus - Cahiers Du Libre Avenir,Prêtre autrement.
59835,Chambre Paillasson En Forme De Coeur Tapis Flu...,Chambre Paillasson en forme de coeur Tapis Tap...
23220,2pcs En Alliage D'aluminium Portail Du Carter ...,2pcs en alliage d&#39;aluminium Portail du car...
36107,Harnais Chien Arnais Noir Anti Traction Gilet ...,<p><b>La description:</b></p><br /><p> Fait de...
...,...,...
2975,Balise De Jardin Koral 1 Ampoule 60w,Coloris : Gris Urbain <br />Matèriau : Acier i...
38126,¿¿Tui De Protection En Silicone Souple R¿¿Sist...,¿¿tui de protection en silicone souple r¿¿sist...
10515,Rage The Werewolf: The Apocalypse Tradin...,None
82324,Pour Amuser Les Coccinelles,None


In [ ]:
# Remplacer les NaN de 'description' par une chaîne vide
X_train['description'] = X_train['description'].fillna('')

# Concaténer les deux colonnes
# Le jeton [SEP] est optionnel mais peut aider
X_train['full_text'] = X_train['designation'] + ' [SEP] ' + X_train['description']

NameError: name 'X_train' is not defined

In [ ]:
#TODO
# --- Couche de preprocessing qui sera intégrée au modèle ---
# Elle gère la tokenisation (texte -> entiers), la mise en minuscules, etc.
from tensorflow.keras import layers
text_vectorizer = layers.TextVectorization(
    max_tokens=20000, # Taille du vocabulaire
    output_sequence_length=200 # Longueur max des phrases (tronque/padde)  TODO: pick
)
# TODO: adapter text_vectorizer sur données texte une seule fois :
# text_vectorizer.adapt(X_train['full_text'].values)

In [ ]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)  #TODO

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [18]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [19]:
new_preprocessors

{}

In [20]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [21]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [22]:
from tensorflow import keras

### Load or create model

In [ ]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    subversion = last_experiment.get('subversion', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    subversion = last_experiment.get('subversion', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model = define_model(text_vectorizer=text_vectorizer, num_classes=27)


Création d'un nouveau modèle.


In [24]:
if not load_model:
    subversion = int(input(f"subversion (architecture)? (last: {subversion})"))

In [25]:
subversion

3

### Summary

In [26]:
# model.summary()

## Callbacks

### ModelCheckpoint

In [27]:
# # Pick an available filename to save a model.
# subversion=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{subversion}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     subversion+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{subversion}.h5')
# new_location_for_saving_model


In [28]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [29]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

In [30]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [31]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_loss', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='min',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [32]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [33]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [34]:
import datetime
tensor_board_folder = artifacts_folder / "tensorboard_logs"
tensor_board_folder_timestamp = tensor_board_folder / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder_timestamp,
    histogram_freq=1 # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [35]:
import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793

In [36]:
# max_epochs=13

# # Calculate expected duration
# available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
# available_minutes

In [37]:
# Pick max_epochs based on your available time
available_minutes=20

max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

9

### compilation and callbacks

In [38]:
learning_rate=0.001

In [39]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [40]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [41]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=20, max_epochs=9, champion_path=None ?

In [ ]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=max(total_epochs_trained-1,0), callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 0/9


2025-10-03 16:15:46.630829: I external/local_xla/xla/service/service.cc:163] XLA service 0x759edc012d90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-03 16:15:46.630846: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-03 16:15:47.084695: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-03 16:15:48.421600: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-03 16:15:49.349657: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-03 16:15:49.

212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step - accuracy: 0.1965 - loss: 3.1310

2025-10-03 16:16:53.028318: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12305', 12 bytes spill stores, 12 bytes spill loads

2025-10-03 16:16:57.013506: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 16:16:57.107579: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 16:16:57.717753: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 16:16:57.817293: E external/local_xla/xla/s

213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.1969 - loss: 3.1294

2025-10-03 16:18:17.218062: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-03 16:18:25.227827: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 16:18:25.326219: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 16:18:26.177309: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

213/213 ━━━━━━━━━━━━━━━━━━━━ 174s 668ms/step - accuracy: 0.2849 - loss: 2.7960 - val_accuracy: 0.4520 - val_loss: 2.1860 - learning_rate: 0.0010
Epoch 1/9
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 482ms/step - accuracy: 0.4343 - loss: 2.2136 - val_accuracy: 0.5053 - val_loss: 1.9656 - learning_rate: 0.0010
Epoch 2/9
 57/213 ━━━━━━━━━━━━━━━━━━━━ 27s 175ms/step - accuracy: 0.4692 - loss: 2.0805

## Evaluation

In [ ]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_images/deep_learning/v1/tensor_board_logs'


In [ ]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['val_accuracy'] = max(model_history.history['val_accuracy'])
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.5846090316772461,
 'actual_epochs': 7,
 'minutes_per_epoch': 3.8673305687450226}

In [ ]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [ ]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

531/531 ━━━━━━━━━━━━━━━━━━━━ 74s 130ms/step


array([[6.4887403e-04, 2.0901163e-03, 4.6690591e-03, ..., 6.1272145e-03,
        1.8308549e-04, 3.7787642e-04],
       [1.2739110e-03, 3.2442934e-03, 3.4947013e-03, ..., 2.3647062e-02,
        4.0979846e-04, 2.0867984e-03],
       [2.4760200e-04, 8.9100851e-03, 3.9757214e-02, ..., 1.5164275e-01,
        5.7821861e-04, 1.9715992e-03],
       ...,
       [3.7029502e-01, 1.3788496e-02, 4.5598470e-04, ..., 2.9844852e-04,
        4.2620796e-01, 5.7949522e-03],
       [7.1058673e-04, 2.1386829e-03, 3.9583035e-03, ..., 2.4075380e-02,
        2.6687339e-04, 7.4863172e-04],
       [7.5020073e-03, 6.3158059e-03, 1.4083970e-02, ..., 6.4948119e-02,
        4.5849583e-03, 6.7881559e-04]], shape=(16984, 27), dtype=float32)

In [ ]:
from sklearn import metrics

In [ ]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [ ]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1280,1281,1300,1301,1302,1320,1560,1920,1940,2060,2220,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,,,,
10,389,16,0,0,3,2,5,1,3,0,0,5,0,1,1,3,0,48,104,0,24,0,0,0,17,1
40,33,251,9,2,10,11,16,4,34,0,5,5,5,2,0,10,0,22,39,16,14,0,5,1,6,2
50,1,10,76,16,20,1,26,0,65,0,4,17,6,1,1,11,0,1,12,18,23,0,24,3,0,0
60,1,8,10,98,0,4,13,0,14,0,0,0,0,0,0,1,0,0,0,8,6,0,2,0,0,1
1140,9,19,0,0,362,3,52,1,8,0,2,20,1,8,0,6,0,2,15,3,12,0,6,0,3,2
1160,19,24,0,0,16,660,6,1,0,0,0,0,0,0,0,2,0,10,40,5,6,0,1,0,1,0
1180,12,8,1,0,58,2,17,0,4,0,0,5,1,1,0,5,0,2,19,2,7,0,4,0,3,2
1280,3,8,6,1,90,1,460,2,163,0,23,28,21,7,2,89,0,1,16,3,25,1,19,1,1,3
1281,11,24,0,0,9,13,122,24,11,0,11,4,2,0,6,38,0,4,47,5,48,1,16,3,5,10


In [ ]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.56384076070673, 0.0)

In [ ]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [ ]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

,precision,recall,f1-score,support
10,0.440544,0.624398,0.516600,623.000000
40,0.542117,0.500000,0.520207,502.000000
50,0.506667,0.226190,0.312757,336.000000
60,0.790323,0.590361,0.675862,166.000000
1140,0.550152,0.677903,0.607383,534.000000
1160,0.919220,0.834387,0.874751,791.000000
1180,0.000000,0.000000,0.000000,153.000000
1280,0.359937,0.472279,0.408526,974.000000
1281,0.558140,0.057971,0.105033,414.000000
1300,0.578439,0.771060,0.661003,1009.000000


In [ ]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.597935,0.491483,0.495804,629.037037
std,0.207877,0.272196,0.239801,420.759339
min,0.000000,0.000000,0.000000,153.000000
25%,0.503333,0.231108,0.346379,310.000000
50%,0.563392,0.589163,0.520207,534.000000
75%,0.748659,0.675908,0.668432,953.500000
max,0.974359,0.902299,0.874751,2042.000000


In [ ]:
# negative correlation between support and another measure would suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.341649,0.512187,0.001002
recall,0.341649,1.000000,0.960576,0.090840
f1-score,0.512187,0.960576,1.000000,0.074077
support,0.001002,0.090840,0.074077,1.000000


In [ ]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.56384076070673

In [ ]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.5846090316772461,
 'actual_epochs': 7,
 'minutes_per_epoch': 3.8673305687450226,
 'weighted_avg_f1_score': 0.56384076070673,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2398006328355822)}

## Update tracker

In [ ]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Nouveau champion ! Sauvegarde du modèle.")
    keep_candidate=True
    subversion=last_experiment.get('subversion',1)

    best_epoch_in_session_idx = np.argmin(model_history.history['val_loss'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_loss = model_history.history['val_loss'][best_epoch_in_session_idx]


    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_sv-{subversion}_epoch_index-{best_epoch_global:02d}_val_loss-{best_val_loss:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_filename)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print("Effacement de l'ancien modèle de la même subversion {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le champion.")
    keep_candidate=False


Nouveau champion ! Sauvegarde du modèle.
best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras


In [ ]:
tracker['total_epochs'] = total_epochs_trained + len(model_history.epoch)

In [ ]:
to_track=['subversion','rebalance_with_weights','BATCH_SIZE','learning_rate']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model)
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.5846090316772461,
 'actual_epochs': 7,
 'minutes_per_epoch': 3.8673305687450226,
 'weighted_avg_f1_score': 0.56384076070673,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2398006328355822),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras',
 'total_epochs': 8,
 'subversion': 3,
 'max_epochs': 9,
 'rebalance_with_weights': False,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16}}

In [ ]:
tracker['comment']="Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings."

In [ ]:
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.5846090316772461,
 'actual_epochs': 7,
 'minutes_per_epoch': 3.8673305687450226,
 'weighted_avg_f1_score': 0.56384076070673,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2398006328355822),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras',
 'total_epochs': 8,
 'subversion': 3,
 'max_epochs': 9,
 'rebalance_with_weights': False,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16},
 'comment': 'Increased frac from 0.1 to 0.3.'}

In [ ]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [ ]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [ ]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [ ]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, log_file_path=log_file_path)

Log pour l'expérience subversion 3 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment,best_model_path
0,1,False,20380,32,2.657786,8,13,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.,None
1,2,False,6793,32,1.998601,19,10,0.001,0.569713,0.528593,0.000000,0.247884,256_128_64_32,16_16,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/best_model_sv-2_epochs-10_f1-0.5286.keras
2,2,False,20380,32,3.867331,9,8,0.001,0.584609,0.563841,0.000000,0.239801,256_128_64_32,16_16,Increased frac from 0.1 to 0.3.,artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras


In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
